# Complete DSA Practical (Sections 1–24) — ADR Severity Research Demonstration
## Development of an Integrated Context-Aware Machine Learning Framework for Adverse Drug Reaction Detection and Severity Stratification

**Google Colab:** Run cells in order. Section 15 uploads the supplied CSV. Sections 1–14 establish the original RidgeCV practical using **the same ADR demonstration dataset**, not the diabetes example. Sections 15–24 complete the research ablation, metrics, visualizations, and exports.

**Scientific integrity:** The supplied 360 reports are **synthetic**, not actual patient data. Severity is generated from contextual variables, so any advantage for the context model is built into the simulation. No ADR-negative reports are provided; ADR detection is therefore **not measured** here. This is a teaching/prototype demonstration only.

## 1. Imports, environment and reproducibility

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
SEED=42
np.random.seed(SEED)
print('Pandas:',pd.__version__)
print('Random seed:',SEED)
print('Dataset provenance: SYNTHETIC — teaching only')

## 2. Load the ADR dataset (local file or Colab upload)

In [ ]:
import io, os
DATAFILE='ADR_SYNTHETIC_DEMONSTRATION_DATA.csv'
if os.path.exists(DATAFILE):
    df=pd.read_csv(DATAFILE)
else:
    try:
        from google.colab import files
        print('Upload the provided ADR_SYNTHETIC_DEMONSTRATION_DATA.csv')
        uploaded=files.upload()
        assert uploaded, 'CSV required'
        df=pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
    except ImportError:
        raise FileNotFoundError('Put the supplied CSV beside the notebook or run in Colab.')
print('Rows and columns:',df.shape)
display(df.head())

## 3. Inspect schema, missing values, class balance

In [ ]:
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().rename('missing').to_frame())
display(df.severity.value_counts().rename('records').to_frame())
assert {'severity','severity_score','narrative'}.issubset(df.columns)
print('All labels are synthetic; severity_score encodes Mild=1, Moderate=2, Severe=3.')

## 4. Define predictors and explain target selection

In [ ]:
numeric=['age_years','concomitant_medication_count','dose_mg','dose_mg_per_kg','time_to_onset_days','duration_of_use_days']
X=df[numeric].copy();y=df['severity_score'].astype(float)
print('Predictors:',numeric)
print('Regression target: synthetic ordinal severity score (1, 2, 3)')
print('Excluded to prevent target leakage: severity, severity_score from features; report_id and outcome descriptions.')

## 5. Descriptive statistics and correlation heatmap

In [ ]:
display(X.describe().round(2))
corr_all=X.corr()
plt.figure(figsize=(9,6));sns.heatmap(corr_all,annot=True,cmap='coolwarm',vmin=-1,vmax=1,fmt='.2f');plt.title('Illustrative correlations — synthetic predictor data');plt.tight_layout();plt.show()
strong=corr_all.where(np.triu(np.ones(corr_all.shape),1).astype(bool)).stack()
print('Pairs with |Pearson r| >= 0.70:')
display(strong[strong.abs()>=.7].rename('r').to_frame())

## 6. Visualize the severity target distribution

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
df.severity.value_counts().reindex(['Mild','Moderate','Severe']).plot.bar(ax=ax)
ax.set(title='Synthetic severity distribution',xlabel='Class',ylabel='Reports');plt.tight_layout();plt.show()

## 7. Split training and test data before tuning

In [ ]:
train_idx,test_idx=train_test_split(np.arange(len(df)),test_size=.25,random_state=SEED,stratify=df.severity)
X_train=X.iloc[train_idx];X_test=X.iloc[test_idx]
y_train=y.iloc[train_idx];y_test=y.iloc[test_idx]
print('Training:',len(X_train),'Test:',len(X_test),'Test set held out from tuning')

## 8. Ordinary Linear Regression baseline

In [ ]:
def make_regressor(est):
    return Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',est)])
ols=make_regressor(LinearRegression()).fit(X_train,y_train)
ols_pred=ols.predict(X_test)
print('OLS MSE:',round(mean_squared_error(y_test,ols_pred),4))

## 9. Define two alpha grids and five-fold cross-validation

In [ ]:
narrow=np.linspace(.1,2.0,20)
wide=np.logspace(-4,4,33)
cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
print('Narrow grid:',narrow)
print('Wide logarithmic grid:',wide)
print('All tuning uses training folds only.')

## 10. Train RidgeCV for each alpha grid

In [ ]:
ridge_narrow=make_regressor(RidgeCV(alphas=narrow,cv=cv,scoring='neg_mean_squared_error')).fit(X_train,y_train)
ridge_wide=make_regressor(RidgeCV(alphas=wide,cv=cv,scoring='neg_mean_squared_error')).fit(X_train,y_train)
print('Selected narrow alpha:',ridge_narrow.named_steps['model'].alpha_)
print('Selected wide alpha:',ridge_wide.named_steps['model'].alpha_)

## 11. Evaluate all models: MSE, RMSE, MAE and R²

In [ ]:
base_models={'Linear Regression':ols,'RidgeCV narrow':ridge_narrow,'RidgeCV wide log':ridge_wide}
rows=[]
for name,model in base_models.items():
    pred=model.predict(X_test)
    mse=mean_squared_error(y_test,pred)
    rows.append({'model':name,'alpha':getattr(model.named_steps['model'],'alpha_',np.nan),'MSE':mse,'RMSE':np.sqrt(mse),'MAE':mean_absolute_error(y_test,pred),'R2':r2_score(y_test,pred)})
initial_metrics=pd.DataFrame(rows)
display(initial_metrics.round(4))

## 12. Actual versus predicted and residual plots

In [ ]:
pred=ridge_wide.predict(X_test)
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(y_test,pred,alpha=.65);axes[0].plot([1,3],[1,3],'k--');axes[0].set(xlabel='Actual encoded severity',ylabel='Predicted score',title='Ridge: actual vs predicted')
axes[1].scatter(pred,y_test-pred,alpha=.65);axes[1].axhline(0,color='black',ls='--');axes[1].set(xlabel='Predicted score',ylabel='Residual',title='Residual plot')
plt.tight_layout();plt.show()
print('Interpret cautiously: severity is ordinal, not a validated continuous outcome.')

## 13. Ridge coefficient shrinkage across alpha values

In [ ]:
alpha_path=np.logspace(-4,4,65)
scaled=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]).fit_transform(X_train)
coef_path=np.array([Ridge(alpha=a).fit(scaled,y_train).coef_ for a in alpha_path])
plt.figure(figsize=(10,5))
for i,col in enumerate(numeric):plt.semilogx(alpha_path,coef_path[:,i],label=col)
plt.axhline(0,color='black',lw=.7);plt.xlabel('Alpha (log scale)');plt.ylabel('Standardized coefficient');plt.title('Coefficient shrinkage on training data');plt.legend(bbox_to_anchor=(1.02,1),loc='upper left');plt.tight_layout();plt.show()

## 14. Cross-validation error curve and interpretation

In [ ]:
cv_mse=np.array([-cross_val_score(make_regressor(Ridge(alpha=a)),X_train,y_train,cv=cv,scoring='neg_mean_squared_error').mean() for a in alpha_path])
plt.figure(figsize=(9,4));plt.semilogx(alpha_path,cv_mse)
plt.axvline(ridge_narrow.named_steps['model'].alpha_,ls='--',label='Narrow best')
plt.axvline(ridge_wide.named_steps['model'].alpha_,ls=':',label='Wide best')
plt.xlabel('Alpha');plt.ylabel('Mean cross-validation MSE');plt.title('Alpha tuning (training folds)');plt.legend();plt.tight_layout();plt.show()
print('Ridge generally reduces coefficient magnitudes as alpha grows; excessively high alpha may underfit.')

## 15. Upload the ADR CSV dataset into Colab

In [ ]:
# Section 15: Use the same uploaded CSV from Section 2; optionally replace it.
print('Dataset already loaded:',df.shape)
print('To replace it, rerun Section 2 and upload a new CSV, then rerun all downstream cells.')

## 16. Validate data and define prediction tasks

In [ ]:
required=['report_id','narrative','age_years','patient_sex','comorbidity','concomitant_medication_count','dose_mg','dose_mg_per_kg','time_to_onset_days','duration_of_use_days','severity','severity_score']
assert not (set(required)-set(df.columns)), f'Missing: {set(required)-set(df.columns)}'
assert df.report_id.is_unique, 'Duplicate IDs: investigate before modeling'
assert df.severity.isin(['Mild','Moderate','Severe']).all()
assert (df.severity_score==df.severity.map({'Mild':1,'Moderate':2,'Severe':3})).all()
print('Dataset provenance: SYNTHETIC — labels generated, not clinician adjudicated')
print('Severity distribution:');display(df.severity.value_counts().to_frame('count'))
print('Missingness:');display(df.isna().sum().to_frame('missing'))
print('Important: severity_score is directly encoded from severity; regression is a teaching proxy, not an independent endpoint.')


## 17. EDA and correlation heatmap (train only)

In [ ]:
numeric=['age_years','concomitant_medication_count','dose_mg','dose_mg_per_kg','time_to_onset_days','duration_of_use_days']
train_idx,test_idx=train_test_split(np.arange(len(df)),test_size=.25,random_state=SEED,stratify=df.severity)
train=df.iloc[train_idx].copy();test=df.iloc[test_idx].copy()
print('Train:',len(train),'Test:',len(test));print('Test class counts:',test.severity.value_counts().to_dict())
corr=train[numeric].corr();display(corr.round(3))
plt.figure(figsize=(10,7));sns.heatmap(corr,annot=True,fmt='.2f',cmap='coolwarm',vmin=-1,vmax=1);plt.title('Training-only Pearson correlations — SYNTHETIC ADR data');plt.tight_layout();plt.show()
upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().sort_values(key=abs,ascending=False)
print('Ranked predictor correlations:');display(upper.rename('Pearson r').to_frame())
print('Strong correlations |r| >= 0.70:');display(upper[upper.abs()>=.7].rename('Pearson r').to_frame())


## 18. Linear Regression and RidgeCV: narrow and wide alpha grids

In [ ]:
Xtr=train[numeric];Xte=test[numeric];ytr=train.severity_score.astype(float);yte=test.severity_score.astype(float)
narrow=np.linspace(.1,2,20);wide=np.logspace(-4,4,33)
cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
def regpipe(model):return Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',model)])
models={'Linear Regression':regpipe(LinearRegression()),'RidgeCV narrow':regpipe(RidgeCV(alphas=narrow,cv=cv,scoring='neg_mean_squared_error')),'RidgeCV wide log':regpipe(RidgeCV(alphas=wide,cv=cv,scoring='neg_mean_squared_error'))}
regrows=[];regpred={}
for name,model in models.items():
    model.fit(Xtr,ytr);pred=model.predict(Xte);regpred[name]=pred
    regrows.append({'model':name,'selected_alpha':getattr(model.named_steps['model'],'alpha_',np.nan),'MSE':mean_squared_error(yte,pred),'RMSE':np.sqrt(mean_squared_error(yte,pred)),'MAE':mean_absolute_error(yte,pred),'R2':r2_score(yte,pred)})
regresults=pd.DataFrame(regrows);display(regresults.round(4))
print('Narrow selected alpha:',models['RidgeCV narrow'].named_steps['model'].alpha_)
print('Wide selected alpha:',models['RidgeCV wide log'].named_steps['model'].alpha_)
print('NOTE: these are ordinal severity-score proxies, not validated continuous clinical outcomes.')


## 19. Visualize alpha tuning and coefficient shrinkage

In [ ]:
from sklearn.model_selection import cross_val_score
alpha_path=np.logspace(-4,4,65)
cv_errors=[]
for a in alpha_path:
    m=regpipe(Ridge(alpha=a))
    cv_errors.append(-cross_val_score(m,Xtr,ytr,cv=cv,scoring='neg_mean_squared_error').mean())
plt.figure(figsize=(9,4));plt.semilogx(alpha_path,cv_errors);plt.axvline(models['RidgeCV narrow'].named_steps['model'].alpha_,ls='--',label='Narrow selected');plt.axvline(models['RidgeCV wide log'].named_steps['model'].alpha_,ls=':',label='Wide selected');plt.xlabel('Alpha (log scale)');plt.ylabel('Mean 5-fold CV MSE');plt.title('Training-only CV error vs alpha');plt.legend();plt.tight_layout();plt.show()
pre=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]);Z=pre.fit_transform(Xtr)
coefs=np.vstack([Ridge(alpha=a).fit(Z,ytr).coef_ for a in alpha_path])
plt.figure(figsize=(10,5))
for j,col in enumerate(numeric):plt.semilogx(alpha_path,coefs[:,j],label=col)
plt.axhline(0,color='black',lw=.7);plt.xlabel('Alpha (log scale)');plt.ylabel('Standardized coefficient');plt.title('Ridge coefficient shrinkage — SYNTHETIC data');plt.legend(bbox_to_anchor=(1.02,1),loc='upper left');plt.tight_layout();plt.show()
print('Coefficient L2 norm: first alpha',round(np.linalg.norm(coefs[0]),4),'last alpha',round(np.linalg.norm(coefs[-1]),4))


## 20. Proposed novelty demonstration: text-only vs text + clinical context

In [ ]:
categorical=['patient_sex','comorbidity']
# Identical rows, classifier and tuning for the controlled ablation.
def severity_model(with_context):
    transformers=[('text',TfidfVectorizer(ngram_range=(1,2),max_features=3000,min_df=2),'narrative')]
    if with_context:
        transformers += [('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)]
    return Pipeline([('features',ColumnTransformer(transformers)),('classifier',LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED))])
classrows=[];preds={};trained={}
for name,context in [('Text-only',False),('Text + structured context',True)]:
    model=severity_model(context);model.fit(train,train.severity);pred=model.predict(test);preds[name]=pred;trained[name]=model
    classrows.append({'model':name,'macro_F1':f1_score(test.severity,pred,average='macro',zero_division=0),'balanced_accuracy':balanced_accuracy_score(test.severity,pred),'accuracy':accuracy_score(test.severity,pred)})
    print('\n',name,'\n',classification_report(test.severity,pred,zero_division=0))
classresults=pd.DataFrame(classrows);display(classresults.round(4))
delta=classresults.iloc[1].macro_F1-classresults.iloc[0].macro_F1
print('Context minus text-only macro-F1:',round(delta,4))
print('INTERPRETATION: SYNTHETIC labels depend on context by construction; this is a functionality demonstration, not empirical evidence of novel clinical performance.')


## 21. Confusion matrices and comparison charts

In [ ]:
labels=['Mild','Moderate','Severe']
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,(name,pred) in zip(axes,preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(test.severity,pred,labels=labels),display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False);ax.set_title(name)
plt.suptitle('Held-out severity classification — SYNTHETIC data');plt.tight_layout();plt.show()
fig,axes=plt.subplots(1,2,figsize=(11,4))
classresults.set_index('model')[['macro_F1','balanced_accuracy']].plot.bar(ax=axes[0],rot=15,ylim=(0,1),title='Classification comparison')
regresults.set_index('model')[['MSE','MAE']].plot.bar(ax=axes[1],rot=15,title='Regression error comparison')
plt.tight_layout();plt.show()


## 22. Robustness check: repeated stratified validation of novelty ablation

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
splitter=RepeatedStratifiedKFold(n_splits=3,n_repeats=5,random_state=SEED)
rows=[]
for fold,(a,b) in enumerate(splitter.split(df,df.severity),1):
    parttrain=df.iloc[a];parttest=df.iloc[b];record={'fold':fold}
    for name,ctx in [('text',False),('context',True)]:
        m=severity_model(ctx);m.fit(parttrain,parttrain.severity)
        record[name]=f1_score(parttest.severity,m.predict(parttest),average='macro',zero_division=0)
    record['delta']=record['context']-record['text'];rows.append(record)
robust=pd.DataFrame(rows)
display(robust.describe().round(4))
print('Mean paired macro-F1 difference:',round(robust.delta.mean(),4))
print('Standard deviation across folds:',round(robust.delta.std(),4))
print('Repeated folds overlap and are not independent clinical cohorts. Synthetic results cannot establish generalization.')


## 23. Export CSV tables, figures, and a presentation summary

In [ ]:
from pathlib import Path
out=Path('ADR_demo_results');out.mkdir(exist_ok=True)
regresults.to_csv(out/'ridge_regression_metrics.csv',index=False)
classresults.to_csv(out/'severity_ablation_metrics.csv',index=False)
robust.to_csv(out/'repeated_cv_ablation.csv',index=False)
upper.rename('pearson_r').to_csv(out/'predictor_correlations.csv')
report=f"""# ADR Practical: presentation results (SYNTHETIC DATA ONLY)
Dataset: {len(df)} generated records. Training: {len(train)}, held-out test: {len(test)}.

## Research question
Does structured clinical context improve severity classification over text-only baseline?

## Ablation
{classresults.to_string(index=False)}
Paired held-out macro-F1 difference: {delta:+.4f}.
Repeated stratified CV mean delta: {robust.delta.mean():+.4f}.

## DSA assignment: Linear Regression vs RidgeCV
{regresults.to_string(index=False)}
Narrow alpha: {models['RidgeCV narrow'].named_steps['model'].alpha_}
Wide alpha: {models['RidgeCV wide log'].named_steps['model'].alpha_}

## Strongest correlations
{upper.head(6).to_string()}

## Interpretations
Correlation measures linear association, not causation. Ridge coefficient magnitudes generally shrink toward zero as alpha increases; too much regularization may underfit. The regression target is an ordinal encoding of severity.

## NOVELTY CLAIM BOUNDARY
This is a synthetic, generated-label demonstration; context influences labels by construction. The results establish that the proposed evaluation pipeline can run, not that it improves ADR care. No ADR-vs-non-ADR detection is evaluated. Clinical claims require real, appropriately licensed, independently annotated data and external validation.
"""
(out/'presentation_summary.md').write_text(report,encoding='utf-8');print(report)
try:
    from google.colab import files
    import shutil
    archive=shutil.make_archive('ADR_demo_results','zip',out)
    files.download(archive)
except ImportError:pass


## 24. Suggested oral conclusion and next steps

**Say:** “I adapted the original DSA regression practical to an ADR-themed dataset. I compared Linear Regression against RidgeCV with narrow and wide alpha searches, interpreted predictor correlations and coefficient shrinkage, and added a controlled text-only versus context-aware severity ablation. These demonstrations show the **proposed evaluation methodology**, not validated clinical effectiveness, because the supplied records and labels are synthetic.”

**To establish actual novelty:** Replace this synthetic CSV with an ethically sourced, appropriately licensed, independently annotated ADR cohort; use patient-level splits, an independent test cohort, proper ADR-negative examples for detection, gold-standard severity adjudication, calibration, confidence intervals, and comparison with strong baselines. Do not mix unrelated records or report these synthetic scores as clinical evidence.

**Assignment checklist:** CSV upload ✓; data inspection ✓; correlation heatmap ✓; high-correlation pairs ✓; linear/RidgeCV narrow/RidgeCV wide ✓; selected alpha ✓; MSE/RMSE/MAE/R² ✓; shrinkage ✓; novelty ablation ✓; confusion matrices ✓; saved results ✓.